# Carga no Autonomous Database · Conecta Saúde

Este notebook **gera os scripts SQL** que criam e carregam o banco no Oracle Autonomous Database. Ele não se conecta ao banco: produz arquivos `.sql` que você revisa e executa no SQL Developer Web, no SQLcl ou no Database Actions.

Essa separação é deliberada. Um notebook que executa DDL direto no banco esconde o que fez — e num trabalho avaliado, o script revisável **é** parte da entrega.

---

## O caminho dos dados

```text
dados/tratado/*.parquet          (local, gerado pelos notebooks de extração)
        ↓  upload manual
Object Storage do OCI            (bucket, um prefixo por fonte)
        ↓  DBMS_CLOUD.CREATE_EXTERNAL_TABLE
Camada BRONZE — tabelas externas  (o dado continua no Object Storage)
        ↓  CREATE TABLE ... AS SELECT
Camada SILVER — tabelas internas  (dado dentro do banco, indexado)
        ↓  CREATE VIEW
Camada GOLD — visões do índice    (o ICPA pronto para o dashboard)
```

**Por que duas camadas e não uma.** A tabela externa lê o Parquet direto do Object Storage a cada consulta: é ótima para conferir a carga, mas cada `SELECT` paga o custo de rede. A tabela interna copia o dado uma vez e depois responde rápido, aceita índice e entra em consulta do Select AI sem latência. As duas juntas custam pouco — o conjunto todo tem 1,8 milhão de linhas e cerca de 16 MB.

## Por que os comentários importam mais do que parecem

O script `04_comentarios.sql` gera `COMMENT ON TABLE` e `COMMENT ON COLUMN` para todas as colunas, a partir do dicionário de dados.

Isso não é documentação decorativa: o **Select AI lê esses comentários** para traduzir pergunta em linguagem natural para SQL. Sem eles, o modelo vê `QT_SUS` e precisa adivinhar; com eles, lê "leitos disponíveis ao SUS, subconjunto de QT_EXIST" e acerta a consulta. É o item de maior retorno por linha de código deste notebook.

## 1. Pré-requisitos no OCI

Antes de rodar os scripts gerados, três coisas precisam existir do lado da Oracle.

### Um bucket no Object Storage

Com os Parquet organizados por fonte, espelhando a pasta local:

```text
conecta-saude/
├── cnes/     cnes_lt_*.parquet, cnes_st_*.parquet, cnes_silver_*.parquet
├── ibge/     ibge_municipios_populacao_2024.parquet
├── sigtap/   sigtap_*.parquet
├── sih/      sih_silver_*.parquet
└── sia/      sia_silver_*.parquet
```

### Um Auth Token

Em **Identity → Users → seu usuário → Auth Tokens → Generate Token**. Ele aparece **uma única vez** — copie na hora. É a senha que o `DBMS_CLOUD.CREATE_CREDENTIAL` usa.

### O namespace do tenancy

Em **Object Storage → Bucket Details**, campo *Namespace*. É uma cadeia curta como `grxxxxxxxxxx`, diferente do nome do tenancy.

> **Sobre o token no script.** O `01_credencial.sql` sai com um marcador `SUBSTITUA_PELO_AUTH_TOKEN`, não com o segredo. Preencha na hora de executar e não faça commit do arquivo preenchido — a pasta `sql/` não está no `.gitignore`, então um token colado ali iria para o GitHub.

In [ ]:
# Só pandas e pyarrow. Este notebook não se conecta ao banco.
%pip install -q pandas pyarrow

## 2. Parâmetros da carga

Ajuste as quatro primeiras variáveis com os dados do seu tenancy.

In [ ]:
import re
from collections import defaultdict
from datetime import date
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------ OCI
NAMESPACE = "SUBSTITUA_PELO_NAMESPACE"
BUCKET = "conecta-saude"
REGIAO = "sa-saopaulo-1"
USUARIO_OCI = "SUBSTITUA_PELO_USUARIO"      # ex.: oracleidentitycloudservice/voce@email.com

CREDENCIAL = "CRED_OBJECT_STORAGE"
ESQUEMA = "CONECTA"


def raiz_do_projeto() -> Path:
    '''Devolve a pasta do repositório, subindo até encontrar o .git.'''
    atual = Path.cwd().resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
    return atual


RAIZ = raiz_do_projeto()
DIRETORIO_TRATADO = RAIZ / "dados" / "tratado"
DIRETORIO_SQL = RAIZ / "sql"
DIRETORIO_SQL.mkdir(parents=True, exist_ok=True)

URL_BASE = f"https://objectstorage.{REGIAO}.oraclecloud.com/n/{NAMESPACE}/b/{BUCKET}/o"

print(f"Raiz do projeto : {RAIZ}")
print(f"Scripts em      : {DIRETORIO_SQL}")
print(f"URL base        : {URL_BASE}")

## 3. Descoberta das tabelas e do tipo de cada coluna

As tabelas saem dos arquivos que existem em `dados/tratado/`, e o tipo de cada coluna sai do dado real — não de uma lista escrita à mão que se desatualiza.

### Como cada tipo é escolhido

| Origem | Oracle | Por quê |
|---|---|---|
| texto | `VARCHAR2(n CHAR)` | `n` é o maior comprimento observado, com folga de 20% |
| inteiro | `NUMBER(19)` | cabe qualquer contagem do projeto |
| decimal | `NUMBER` | precisão exata; valores em reais não podem ter erro de ponto flutuante |

**`CHAR` em vez de `BYTE`** é obrigatório aqui: nomes como "OBSTETRÍCIA" têm acentos, que ocupam 2 bytes em UTF-8. Dimensionar por byte truncaria descrições e a carga falharia com `value too large for column`.

**`NUMBER` em vez de `BINARY_DOUBLE`** para os valores monetários: ponto flutuante binário não representa `0,10` exatamente, e somar 14 milhões de AIH acumularia erro visível no total.

In [ ]:
def nome_familia(caminho: Path) -> str:
    '''Reduz o nome do arquivo ao padrão da família, sem UF nem competência.'''
    base = caminho.stem
    base = re.sub(r"_[A-Z]{2}_(?=\d)", "_", base)      # cnes_lt_SP_2024 -> cnes_lt_2024
    base = re.sub(r"_\d{6}$", "", base)
    base = re.sub(r"_\d{4}$", "", base)
    return base


def nome_oracle(texto: str) -> str:
    '''Converte para identificador Oracle: maiúsculo, sem acento, sem símbolo.'''
    limpo = re.sub(r"[^A-Za-z0-9_]", "_", texto).upper().strip("_")
    return re.sub(r"_+", "_", limpo)[:128]


def tipo_oracle(serie: pd.Series) -> str:
    '''Escolhe o tipo Oracle a partir do dado real da coluna.'''
    if pd.api.types.is_integer_dtype(serie):
        return "NUMBER(19)"
    if pd.api.types.is_float_dtype(serie):
        return "NUMBER"
    if pd.api.types.is_bool_dtype(serie):
        return "NUMBER(1)"

    maior = int(serie.astype(str).str.len().max() or 1)
    # Folga de 20%: a competência seguinte pode trazer descrição mais longa,
    # e ampliar coluna depois exige ALTER com a tabela em uso.
    return f"VARCHAR2({max(int(maior * 1.2), 10)} CHAR)"


arquivos = [p for p in sorted(DIRETORIO_TRATADO.glob("**/*.parquet"))
            if p.parent.name != "_dicionario"]

familias = defaultdict(list)
for caminho in arquivos:
    familias[(caminho.parent.name, nome_familia(caminho))].append(caminho)

tabelas = {}
for (fonte, base), caminhos in sorted(familias.items()):
    df = (pd.read_parquet(caminhos[0]) if len(caminhos) == 1
          else pd.concat([pd.read_parquet(c) for c in caminhos], ignore_index=True))
    tabelas[nome_oracle(base)] = {
        "fonte": fonte,
        "arquivos": caminhos,
        "linhas": len(df),
        # O padrão de arquivo vira curinga: 27 arquivos de UF, uma tabela só.
        "padrao": re.sub(r"_[A-Z]{2}_(?=\d)", "_*_", caminhos[0].name),
        "colunas": [(nome_oracle(c), tipo_oracle(df[c]), c) for c in df.columns],
    }

print(f"{len(tabelas)} tabelas, {sum(t['linhas'] for t in tabelas.values()):,} linhas\n")
for nome, t in tabelas.items():
    print(f"  {nome:<38} {t['linhas']:>9,} linhas  {len(t['colunas']):>2} col  "
          f"{len(t['arquivos']):>2} arq  {t['padrao']}")

## 4. Script 1 — credencial de acesso ao Object Storage

O `DBMS_CLOUD.CREATE_CREDENTIAL` guarda no banco o par usuário/token que autoriza o ADB a ler o bucket. É executado uma vez.

In [ ]:
def escrever(nome: str, conteudo: str) -> Path:
    destino = DIRETORIO_SQL / nome
    destino.write_text(conteudo, encoding="utf-8")
    return destino


CABECALHO = f"""--------------------------------------------------------------------------------
-- Conecta Saude — gerado por carga_autonomous_database_conecta_saude.ipynb
-- {date.today().isoformat()}
-- NAO EDITE A MAO: este arquivo e sobrescrito a cada execucao do notebook.
--------------------------------------------------------------------------------
"""

sql = [CABECALHO, f"""
-- Executar uma unica vez, conectado como o usuario {ESQUEMA}.
-- O Auth Token e gerado em Identity > Users > Auth Tokens e aparece so uma vez.

BEGIN
  DBMS_CLOUD.DROP_CREDENTIAL(credential_name => '{CREDENCIAL}');
EXCEPTION
  WHEN OTHERS THEN NULL;   -- ainda nao existe na primeira execucao
END;
/

BEGIN
  DBMS_CLOUD.CREATE_CREDENTIAL(
    credential_name => '{CREDENCIAL}',
    username        => '{USUARIO_OCI}',
    password        => 'SUBSTITUA_PELO_AUTH_TOKEN'
  );
END;
/

-- Conferencia: deve listar a credencial como ENABLED.
SELECT credential_name, username, enabled
  FROM user_credentials
 WHERE credential_name = '{CREDENCIAL}';

-- Conferencia: deve listar os arquivos do bucket.
SELECT object_name, bytes
  FROM DBMS_CLOUD.LIST_OBJECTS('{CREDENCIAL}', '{URL_BASE}/')
 FETCH FIRST 20 ROWS ONLY;
"""]

caminho = escrever("01_credencial.sql", "\n".join(sql))
print(f"{caminho.name}  ({caminho.stat().st_size} bytes)")

## 5. Script 2 — camada Bronze, tabelas externas

Uma tabela externa por família de arquivos. O curinga no nome é o que faz os 27 arquivos de UF virarem uma tabela só:

```text
cnes_lt_*.parquet   →   EXT_CNES_LT
```

O `DBMS_CLOUD.CREATE_EXTERNAL_TABLE` lê o schema do próprio Parquet, então não é preciso declarar coluna por coluna aqui — a declaração explícita fica para a camada Silver.

In [ ]:
sql = [CABECALHO, "-- Camada BRONZE: tabelas externas sobre o Object Storage.\n"
                  "-- O dado continua no bucket; o banco so le sob demanda.\n"]

for nome, t in tabelas.items():
    externa = f"EXT_{nome}"
    sql.append(f"""
--------------------------------------------------------------------------------
-- {externa}   ({t['linhas']:,} linhas em {len(t['arquivos'])} arquivo(s))
--------------------------------------------------------------------------------
BEGIN
  DBMS_CLOUD.DROP_EXTERNAL_TABLE(table_name => '{externa}');
EXCEPTION
  WHEN OTHERS THEN NULL;
END;
/

BEGIN
  DBMS_CLOUD.CREATE_EXTERNAL_TABLE(
    table_name      => '{externa}',
    credential_name => '{CREDENCIAL}',
    file_uri_list   => '{URL_BASE}/{t["fonte"]}/{t["padrao"]}',
    format          => JSON_OBJECT('type' VALUE 'parquet')
  );
END;
/
""")

sql.append("""
-- Conferencia da Bronze: cada contagem deve bater com a coluna 'linhas'
-- impressa pelo notebook na secao 3.
""")
for nome, t in tabelas.items():
    sql.append(f"SELECT '{nome}' AS tabela, COUNT(*) AS linhas, "
               f"{t['linhas']} AS esperado FROM EXT_{nome};")

caminho = escrever("02_bronze_externas.sql", "\n".join(sql))
print(f"{caminho.name}  ({caminho.stat().st_size / 1024:.1f} KB, {len(tabelas)} tabelas externas)")

## 6. Script 3 — camada Silver, tabelas internas

Aqui as colunas são declaradas explicitamente, com o tipo derivado do dado real. A carga é um `INSERT ... SELECT` a partir da externa — mais simples que `COPY_DATA` e com a vantagem de que qualquer erro de tipo aparece na hora.

As chaves de junção recebem índice: são as colunas que todo `JOIN` do dashboard vai usar.

In [ ]:
# Colunas que participam de junção entre fontes e por isso ganham índice.
CHAVES_INDEXAVEIS = {
    "CNES", "CODUFMUN", "MUNIC_RES", "MUNIC_MOV", "PA_UFMUN", "PA_MUNPCN",
    "COD_MUNICIPIO_DATASUS", "COD_MUNICIPIO_IBGE", "CO_PROCEDIMENTO",
    "PROC_REA", "PA_PROC_ID", "COMPETENCIA", "UF",
}

sql = [CABECALHO, "-- Camada SILVER: tabelas internas, carregadas a partir da Bronze.\n"]

for nome, t in tabelas.items():
    colunas = ",\n".join(f"  {c:<28} {tipo}" for c, tipo, _ in t["colunas"])
    sql.append(f"""
--------------------------------------------------------------------------------
-- {nome}   ({t['linhas']:,} linhas)
--------------------------------------------------------------------------------
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE {nome} PURGE';
EXCEPTION
  WHEN OTHERS THEN NULL;
END;
/

CREATE TABLE {nome} (
{colunas}
);

INSERT /*+ APPEND */ INTO {nome}
SELECT {', '.join(c for c, _, _ in t['colunas'])}
  FROM EXT_{nome};

COMMIT;
""")
    for coluna, _, _ in t["colunas"]:
        if coluna in CHAVES_INDEXAVEIS:
            sql.append(f"CREATE INDEX IX_{nome}_{coluna}"[:128] + f" ON {nome} ({coluna});")
    sql.append("")

caminho = escrever("03_silver_tabelas.sql", "\n".join(sql))
print(f"{caminho.name}  ({caminho.stat().st_size / 1024:.1f} KB)")

## 7. Script 4 — comentários para o Select AI

O script que faz a diferença entre um Select AI que acerta e um que chuta.

As descrições vêm do `dicionario_dados.csv`, gerado pelo notebook do dicionário. Se aquele notebook ainda não foi executado, este gera os comentários que puder e avisa quais faltaram.

Um detalhe de implementação que evita erro na execução: aspas simples dentro do texto precisam ser duplicadas, senão o `COMMENT ON` fecha a string no meio da frase e o script quebra.

In [ ]:
caminho_dicionario = DIRETORIO_TRATADO / "_dicionario" / "dicionario_dados.csv"

if caminho_dicionario.exists():
    dicionario = pd.read_csv(caminho_dicionario).fillna("")
    # A chave é o nome da coluna em maiúsculo, igual ao identificador Oracle.
    descricoes = {}
    for linha in dicionario.itertuples():
        chave = nome_oracle(linha.coluna)
        if chave in descricoes:
            continue
        partes = [linha.descricao]
        if linha.observacao:
            partes.append(linha.observacao)
        descricoes[chave] = " ".join(p for p in partes if p).strip()
    print(f"{len(descricoes)} descrições carregadas do dicionário")
else:
    descricoes = {}
    print("Dicionário não encontrado — execute dicionario_de_dados_conecta_saude.ipynb")
    print(f"  esperado em: {caminho_dicionario}")


def texto_sql(valor: str) -> str:
    '''Escapa aspas simples e corta no limite de um comentário Oracle.'''
    return valor.replace("'", "''")[:3900]


DESCRICAO_TABELA = {
    "IBGE_MUNICIPIOS_POPULACAO": "Dimensao territorial: os 5.571 municipios brasileiros com populacao estimada. "
                                 "E a tabela de referencia para todo indicador por habitante.",
    "CNES_LT": "Leitos por estabelecimento, tipo e especialidade, mes a mes.",
    "CNES_ST": "Atributos do estabelecimento de saude: tipo, gestao e natureza juridica, com descricoes.",
    "CNES_SILVER_MUNICIPIO": "Leitos SUS agregados por municipio e competencia.",
    "CNES_SILVER_HOSPITAL": "Leitos SUS agregados por estabelecimento e competencia.",
    "SIH_SILVER_RESIDENCIA": "Internacoes agregadas pelo municipio onde o paciente MORA. "
                             "Cruzada com a populacao, mede necessidade.",
    "SIH_SILVER_ATENDIMENTO": "Internacoes agregadas pelo municipio onde a internacao ACONTECEU. "
                              "Cruzada com os leitos, mede carga sobre a estrutura.",
    "SIH_SILVER_HOSPITAL": "Internacoes agregadas por estabelecimento.",
    "SIH_SILVER_FLUXO": "Internacoes por par municipio de residencia x municipio de atendimento. "
                        "Quando os dois diferem houve deslocamento do paciente.",
    "SIH_SILVER_PROCEDIMENTO": "Internacoes agregadas por procedimento realizado.",
    "SIH_PROCEDIMENTOS_ENRIQUECIDOS": "Procedimentos das internacoes ja com nome e complexidade do SIGTAP.",
    "SIGTAP_PROCEDIMENTOS": "Dimensao de procedimentos do SUS: nome, grupo, complexidade e valor de tabela.",
    "SIA_SILVER_ESTABELECIMENTO": "Producao ambulatorial agregada pelo municipio do estabelecimento.",
    "SIA_SILVER_PACIENTE": "Producao ambulatorial agregada pelo municipio de residencia do paciente.",
    "SIA_SILVER_COMPLEXIDADE": "Producao ambulatorial por municipio e nivel de complexidade.",
}

sql = [CABECALHO,
       "-- Comentarios de tabela e coluna.\n"
       "-- O Select AI usa este texto para traduzir pergunta em SQL: sem ele,\n"
       "-- o modelo ve nomes como QT_SUS e precisa adivinhar o significado.\n"]

sem_descricao = []
for nome, t in tabelas.items():
    comentario = DESCRICAO_TABELA.get(nome, f"Tabela {nome} da fonte {t['fonte'].upper()}.")
    sql.append(f"\nCOMMENT ON TABLE {nome} IS '{texto_sql(comentario)}';")
    for coluna, _, original in t["colunas"]:
        descricao = descricoes.get(coluna, "")
        if not descricao:
            sem_descricao.append(f"{nome}.{coluna}")
            continue
        sql.append(f"COMMENT ON COLUMN {nome}.{coluna} IS '{texto_sql(descricao)}';")

caminho = escrever("04_comentarios.sql", "\n".join(sql))
comentadas = sum(len(t["colunas"]) for t in tabelas.values()) - len(sem_descricao)
total = sum(len(t["colunas"]) for t in tabelas.values())
print(f"{caminho.name}  ({caminho.stat().st_size / 1024:.1f} KB)")
print(f"{comentadas} de {total} colunas comentadas ({100 * comentadas / total:.0f}%)")
if sem_descricao:
    print(f"\nSem descrição no dicionário: {sem_descricao[:8]}")

## 8. Script 5 — camada Gold, as visões do índice

Aqui o modelo de dados vira resposta. Três visões, em ordem crescente de elaboração.

**`VW_CAPACIDADE_MUNICIPIO`** junta população e leitos. Responde "quantos leitos por habitante".

**`VW_DEMANDA_MUNICIPIO`** junta população e internações por residência. Responde "quantas internações por habitante".

**`VW_ICPA_MUNICIPIO`** junta tudo e calcula a pressão. É a tabela que alimenta o dashboard.

### A decisão que atravessa as três

Todas partem do **IBGE com `LEFT JOIN`**, nunca do CNES ou do SIH. Municípios sem leito e sem internação precisam aparecer com zero, e não sumir da lista — são 2.003 municípios sem nenhum leito SUS, onde vivem 14,5 milhões de pessoas. Um `INNER JOIN` produziria um painel que não enxerga exatamente quem mais precisa aparecer.

E o denominador de leitos usa `MUNIC_MOV` (onde a internação ocorreu), não `MUNIC_RES`: os leitos estão onde o hospital está.

In [ ]:
sql = [CABECALHO, """
-- Camada GOLD: visoes analiticas.
-- Todas partem do IBGE com LEFT JOIN para que municipio sem servico apareca
-- como lacuna, e nao como ausencia de linha.
"""]

sql.append(f"""
--------------------------------------------------------------------------------
-- VW_CAPACIDADE_MUNICIPIO — leitos por habitante
--------------------------------------------------------------------------------
CREATE OR REPLACE VIEW VW_CAPACIDADE_MUNICIPIO AS
SELECT
  m.COD_MUNICIPIO_DATASUS,
  m.MUNICIPIO,
  m.UF,
  m.REGIAO,
  m.POPULACAO,
  NVL(l.LEITOS_SUS, 0)                         AS LEITOS_SUS,
  NVL(l.LEITOS_EXISTENTES, 0)                  AS LEITOS_EXISTENTES,
  NVL(l.ESTABELECIMENTOS_COM_LEITO, 0)         AS ESTABELECIMENTOS_COM_LEITO,
  ROUND(NVL(l.LEITOS_SUS, 0) / m.POPULACAO * 10000, 2) AS LEITOS_SUS_POR_10K
FROM IBGE_MUNICIPIOS_POPULACAO m
LEFT JOIN (
  SELECT CODUFMUN,
         AVG(LEITOS_SUS)                 AS LEITOS_SUS,
         AVG(LEITOS_EXISTENTES)          AS LEITOS_EXISTENTES,
         MAX(ESTABELECIMENTOS_COM_LEITO) AS ESTABELECIMENTOS_COM_LEITO
    FROM CNES_SILVER_MUNICIPIO
   GROUP BY CODUFMUN
) l ON l.CODUFMUN = m.COD_MUNICIPIO_DATASUS
WHERE m.POPULACAO > 0;

--------------------------------------------------------------------------------
-- VW_DEMANDA_MUNICIPIO — internacoes por habitante, pelo municipio de residencia
--------------------------------------------------------------------------------
CREATE OR REPLACE VIEW VW_DEMANDA_MUNICIPIO AS
SELECT
  m.COD_MUNICIPIO_DATASUS,
  m.MUNICIPIO,
  m.UF,
  m.REGIAO,
  m.POPULACAO,
  NVL(s.INTERNACOES, 0)      AS INTERNACOES,
  NVL(s.OBITOS, 0)           AS OBITOS,
  NVL(s.DIARIAS_UTI, 0)      AS DIARIAS_UTI,
  NVL(s.VALOR_TOTAL, 0)      AS VALOR_TOTAL,
  ROUND(NVL(s.INTERNACOES, 0) / m.POPULACAO * 10000, 2) AS INTERNACOES_POR_10K,
  ROUND(NVL(s.OBITOS, 0) / NULLIF(s.INTERNACOES, 0) * 100, 2) AS LETALIDADE_PCT
FROM IBGE_MUNICIPIOS_POPULACAO m
LEFT JOIN (
  SELECT MUNIC_RES,
         SUM(INTERNACOES) AS INTERNACOES,
         SUM(OBITOS)      AS OBITOS,
         SUM(DIARIAS_UTI) AS DIARIAS_UTI,
         SUM(VALOR_TOTAL) AS VALOR_TOTAL
    FROM SIH_SILVER_RESIDENCIA
   GROUP BY MUNIC_RES
) s ON s.MUNIC_RES = m.COD_MUNICIPIO_DATASUS
WHERE m.POPULACAO > 0;

--------------------------------------------------------------------------------
-- VW_ICPA_MUNICIPIO — o indice composto
--------------------------------------------------------------------------------
CREATE OR REPLACE VIEW VW_ICPA_MUNICIPIO AS
SELECT
  c.COD_MUNICIPIO_DATASUS,
  c.MUNICIPIO,
  c.UF,
  c.REGIAO,
  c.POPULACAO,
  c.LEITOS_SUS,
  c.LEITOS_SUS_POR_10K,
  d.INTERNACOES,
  d.INTERNACOES_POR_10K,
  d.LETALIDADE_PCT,
  NVL(a.INTERNACOES, 0) AS INTERNACOES_ATENDIDAS,
  -- Internacoes que a estrutura local absorveu por leito instalado.
  -- Municipio sem leito fica NULL: a razao nao existe, e nao e zero.
  ROUND(NVL(a.INTERNACOES, 0) / NULLIF(c.LEITOS_SUS, 0), 1) AS INTERNACOES_POR_LEITO,
  -- Proporcao das internacoes do municipio que ocorreram fora dele.
  ROUND(NVL(f.FORA, 0) / NULLIF(d.INTERNACOES, 0) * 100, 1) AS TAXA_EVASAO_PCT
FROM VW_CAPACIDADE_MUNICIPIO c
JOIN VW_DEMANDA_MUNICIPIO d
  ON d.COD_MUNICIPIO_DATASUS = c.COD_MUNICIPIO_DATASUS
LEFT JOIN (
  SELECT MUNIC_MOV, SUM(INTERNACOES) AS INTERNACOES
    FROM SIH_SILVER_ATENDIMENTO
   GROUP BY MUNIC_MOV
) a ON a.MUNIC_MOV = c.COD_MUNICIPIO_DATASUS
LEFT JOIN (
  SELECT MUNIC_RES, SUM(INTERNACOES) AS FORA
    FROM SIH_SILVER_FLUXO
   WHERE MUNIC_RES <> MUNIC_MOV
   GROUP BY MUNIC_RES
) f ON f.MUNIC_RES = c.COD_MUNICIPIO_DATASUS;

COMMENT ON TABLE VW_ICPA_MUNICIPIO IS
  'Indice Composto de Pressao Assistencial por municipio: populacao, leitos SUS, '
  'internacoes geradas e absorvidas, internacoes por leito e taxa de evasao. '
  'Municipios sem leito aparecem com LEITOS_SUS zero e INTERNACOES_POR_LEITO nulo.';
""")

caminho = escrever("05_gold_views.sql", "\n".join(sql))
print(f"{caminho.name}  ({caminho.stat().st_size / 1024:.1f} KB)")

## 9. Script 6 — validação pós-carga

Nenhuma carga está pronta porque terminou sem erro. Estas consultas comparam o que entrou no banco com o que foi medido localmente — se algum número divergir, a carga tem problema mesmo que nada tenha falhado.

In [ ]:
# Números medidos localmente, para comparar com o que entrar no banco.
referencia = {}
caminho_cnes = DIRETORIO_TRATADO / "cnes" / "cnes_silver_municipio_2024.parquet"
caminho_sih = DIRETORIO_TRATADO / "sih" / "sih_silver_residencia_2024.parquet"
caminho_ibge = DIRETORIO_TRATADO / "ibge" / "ibge_municipios_populacao_2024.parquet"

if caminho_ibge.exists():
    ibge = pd.read_parquet(caminho_ibge)
    referencia["municipios"] = len(ibge)
    referencia["populacao"] = int(ibge["populacao"].sum())
if caminho_sih.exists():
    referencia["internacoes"] = int(pd.read_parquet(caminho_sih)["internacoes"].sum())
if caminho_cnes.exists():
    cnes = pd.read_parquet(caminho_cnes)
    ultima = cnes["COMPETENCIA"].max()
    referencia["leitos_sus_ultima_competencia"] = int(
        cnes[cnes["COMPETENCIA"] == ultima]["leitos_sus"].sum()
    )

sql = [CABECALHO, "-- Validacao pos-carga: cada consulta traz o valor esperado ao lado.\n"]

sql.append("-- Contagem de linhas por tabela.")
for nome, t in tabelas.items():
    sql.append(f"SELECT '{nome}' AS tabela, COUNT(*) AS carregado, "
               f"{t['linhas']} AS esperado FROM {nome};")

sql.append(f"""
-- Totais de negocio, medidos localmente antes da carga.
SELECT 'municipios'  AS metrica, COUNT(*) AS valor,
       {referencia.get('municipios', 0)} AS esperado FROM IBGE_MUNICIPIOS_POPULACAO
UNION ALL
SELECT 'populacao', SUM(POPULACAO),
       {referencia.get('populacao', 0)} FROM IBGE_MUNICIPIOS_POPULACAO
UNION ALL
SELECT 'internacoes', SUM(INTERNACOES),
       {referencia.get('internacoes', 0)} FROM SIH_SILVER_RESIDENCIA;

-- As chaves de juncao precisam casar: estas consultas devem devolver ZERO linhas.
SELECT 'CNES sem municipio no IBGE' AS problema, COUNT(*) AS orfaos
  FROM CNES_SILVER_MUNICIPIO c
 WHERE NOT EXISTS (SELECT 1 FROM IBGE_MUNICIPIOS_POPULACAO m
                    WHERE m.COD_MUNICIPIO_DATASUS = c.CODUFMUN)
HAVING COUNT(*) > 0;

SELECT 'SIH sem municipio no IBGE' AS problema, COUNT(*) AS orfaos
  FROM SIH_SILVER_RESIDENCIA s
 WHERE NOT EXISTS (SELECT 1 FROM IBGE_MUNICIPIOS_POPULACAO m
                    WHERE m.COD_MUNICIPIO_DATASUS = s.MUNIC_RES)
HAVING COUNT(*) > 0;

-- Amostra do indice: os 10 municipios de maior pressao entre os que tem
-- estrutura relevante.
SELECT MUNICIPIO, UF, POPULACAO, LEITOS_SUS, INTERNACOES_POR_LEITO, TAXA_EVASAO_PCT
  FROM VW_ICPA_MUNICIPIO
 WHERE LEITOS_SUS >= 100
 ORDER BY INTERNACOES_POR_LEITO DESC
 FETCH FIRST 10 ROWS ONLY;
""")

caminho = escrever("99_validacao.sql", "\n".join(sql))
print(f"{caminho.name}  ({caminho.stat().st_size / 1024:.1f} KB)")
print("\nValores de referência medidos localmente:")
for chave, valor in referencia.items():
    print(f"  {chave:<32} {valor:>15,}")

## 10. Ordem de execução

Os scripts são numerados na ordem em que devem rodar.

In [ ]:
print("Scripts gerados em", DIRETORIO_SQL)
print()
ORDEM = [
    ("01_credencial.sql", "uma vez", "cria a credencial e confere o acesso ao bucket"),
    ("02_bronze_externas.sql", "a cada recarga", "tabelas externas sobre o Object Storage"),
    ("03_silver_tabelas.sql", "a cada recarga", "tabelas internas, com tipos e indices"),
    ("04_comentarios.sql", "a cada recarga", "COMMENT ON para o Select AI"),
    ("05_gold_views.sql", "a cada recarga", "visoes analiticas do indice"),
    ("99_validacao.sql", "apos cada carga", "confere contagens e chaves"),
]
for nome, quando, o_que in ORDEM:
    arq = DIRETORIO_SQL / nome
    existe = f"{arq.stat().st_size / 1024:>7.1f} KB" if arq.exists() else "  AUSENTE"
    print(f"  {nome:<26} {existe}   {quando:<16} {o_que}")

print("""
Antes de executar:

  1. preencha NAMESPACE e USUARIO_OCI na secao 2 e rode este notebook de novo;
  2. suba os Parquet de dados/tratado/ para o bucket, uma pasta por fonte;
  3. no 01_credencial.sql, troque SUBSTITUA_PELO_AUTH_TOKEN pelo token real
     — e nao faca commit desse arquivo preenchido.
""")

## 11. Limites que valem estar escritos

**O SIA está com uma competência.** As tabelas `SIA_*` cobrem dezembro de 2024, não o ano. O ano completo custaria 40 horas de extração. Ao comparar SIA com SIH no mesmo painel, use proporções entre municípios, não volumes absolutos — os períodos são diferentes.

**O SISREG é sintético.** Se você subir `SINTETICO_regulacao_*.parquet`, a tabela no banco carrega dado simulado. A coluna `ORIGEM_DADO` marca cada linha, e o nome do arquivo carrega o prefixo. Nenhum gráfico dessa fonte pode ser apresentado como retrato do SUS.

**As visões Gold usam a média das competências para os leitos.** Leito abre e fecha ao longo do ano; a média das 12 competências é mais estável que a foto de dezembro. Para uma série temporal mês a mês, consulte `CNES_SILVER_MUNICIPIO` direto, sem passar pela visão.

**Recarga é destrutiva.** Os scripts fazem `DROP TABLE ... PURGE` antes de criar. É o comportamento certo para um pipeline reproduzível, mas significa que qualquer coisa que alguém tenha criado à mão dentro dessas tabelas se perde.